# Resumable chunk qualification on a Colab T4 (slice B)

**Every cell here is a person-run step for Nate.** Nothing in this notebook signs in, stores a secret, discovers data,
or runs on its own. It is a thin driver: all training logic is the reviewed `train_sft.py` resumable mode
(slice A, #587), and every check is a reviewed script beside it.

What it qualifies: that a run cut into chunks across sessions gives the same result as one uninterrupted run, on this
runtime. It uses **invented rows only** (`qualification.py invented-rows`). Real, private or approved-TRAIN rows need
Nate's separate data-placement approval and are out of scope.

Gates that stay with Nate: the account (it must be one Nate owns and uses himself; Google's terms forbid shared
accounts), installing the lock, mounting Drive, and running any training cell.

Known limits, stated before any run:
1. **Drive locking is UNQUALIFIED.** The lease guard is proven on a local filesystem only. Whether two runtimes
   sharing a Drive folder honour it is measured by the probe in step 5, not assumed. Until that probe shows no
   overlap, run one runtime at a time.
2. **fp16 on the T4, `bf16=False`.** Skipped fp16 updates are counted per chunk from the GradScaler and recorded in
   each chunk's manifest (`skipped_updates`, `evidence.amp`).
3. **Storage is measured, not assumed.** Before each chunk the runner checks the store's reported free space against
   three checkpoints plus a margin, using the last committed chunk's measured size (a lower-bound estimate before the
   first chunk). A Drive mount's reported free space is not proven to be the account's real quota.

## 1. Code: upload the reviewed files (person step)
Upload these files from the reviewed commit of `docs/ai-assistant/training/` to `/content/training/` with the Files
panel: `train_sft.py`, `resumable.py`, `sft_tokenize.py`, `colab_env.py`, `qualification.py`, `drive_lock_probe.py`,
`requirements-colab-cuda.lock`. No repository credentials are used here. The cell prints the code hash every chunk is
bound to; compare it with the hash in the PR before going on.

In [ ]:
%cd /content/training
import train_sft
print("code_sha256", train_sft.code_sha256())

## 2. Install the locked CUDA stack (person step, Nate's install gate)
Hash-checked, no dependency resolution at install time. If pip refuses a hash or a wheel, stop: the lock does not fit
this runtime.

In [ ]:
# PERSON-RUN: installs packages into this Colab runtime.
!pip install --require-hashes --no-deps -r requirements-colab-cuda.lock

## 3. Record the environment and check it against the lock
Restart the session after the install first (Runtime, Restart session), then run this. A non-empty problem list means
this runtime is not the locked one: do not start a lineage.

In [ ]:
%cd /content/training
!python colab_env.py --out /content/env.json
import json
ENV = json.load(open("/content/env.json"))
ENV_ID = ENV["environment_id"]
print(ENV_ID, ENV["gpu"], ENV["lock_check"])
assert not ENV["lock_check"]["problems"], "runtime does not match the lock"

## 4. Mount Drive (person step)
Mounting gives this notebook's code access to the whole Drive of the signed-in account, not only the folder below.
Use a dedicated folder. The OAuth prompt is Nate's; no agent handles it.

In [ ]:
# PERSON-RUN: Google's own interactive mount prompt.
from google.colab import drive
drive.mount("/content/drive")
STORE = "/content/drive/MyDrive/fotc-qualification"
!mkdir -p {STORE}

## 5. Drive lock probe (optional, two runtimes at once)
Only needed before ever letting two runtimes near the same store. Run the first cell in two runtimes at the same time,
each with its own label, then the report cell in one. Any overlap means the store is NOT QUALIFIED for that.

In [ ]:
!python drive_lock_probe.py run --store {STORE}/lockprobe --label runtime-a --seconds 300

In [ ]:
!python drive_lock_probe.py report --store {STORE}/lockprobe

## 6. Invented rows

In [ ]:
!python qualification.py invented-rows --out /content/invented.jsonl --rows 64

## 7. Uninterrupted reference run (person step: runs training)
`T` optimizer updates in one chunk. The base model is downloaded from its pinned public revision by `train_sft.py`.

In [ ]:
T, K = 8, 3
!mkdir -p /content/run-whole
!python train_sft.py --data /content/invented.jsonl --run-dir /content/run-whole \
    --resume-store {STORE}/whole --run-id whole --total-steps {T} --chunk-steps {T} \
    --environment-id {ENV_ID} --environment-record /content/env.json

## 8. Chunked run: first chunk (person step: runs training)
`K` updates, then commit. Each training cell is its own Python process, so no training state survives in memory
between chunks; restarting the session (Runtime, Restart session) before step 9 makes that explicit. Rerun steps 3 and 4
after a restart. A **disconnected** runtime loses `/content`; the committed chunks on Drive are what resume uses.

In [ ]:
!mkdir -p /content/run-chunk-1
!python train_sft.py --data /content/invented.jsonl --run-dir /content/run-chunk-1 \
    --resume-store {STORE}/chunked --run-id chunked --total-steps {T} --chunk-steps {K} \
    --environment-id {ENV_ID} --environment-record /content/env.json

## 9. Chunked run: resume to the end (person step: runs training, in a fresh session)

In [ ]:
T, K = 8, 3
!mkdir -p /content/run-chunk-2
!python train_sft.py --data /content/invented.jsonl --run-dir /content/run-chunk-2 \
    --resume-store {STORE}/chunked --run-id chunked --total-steps {T} --chunk-steps {T - K} \
    --environment-id {ENV_ID} --environment-record /content/env.json

## 10. Compare against the gate fixed before results
Matched-step loss within atol 1e-4 / rtol 1e-3, identical learning rate at every step, final adapter relative L2
difference at most 1e-4. A FAIL is investigated, never met by widening a tolerance. The resumed chunk's `loss.jsonl`
carries the restored history of the earlier chunk too, so the comparison still works if `/content/run-chunk-1` was lost
to a disconnect (then drop it from `--chunks`).

In [ ]:
!python qualification.py compare --whole /content/run-whole --chunks /content/run-chunk-1 /content/run-chunk-2 \
    --whole-store {STORE}/whole --chunk-store {STORE}/chunked --whole-run whole --chunk-run chunked